# Lab: Learning Theory and Regularization


## Learning goals

By the end of this lab, you should be able to:

- Distinguish training error, validation error, and test error.
- Use a real dataset to observe overfitting and the bias-variance tradeoff.
- Fit and compare ordinary least squares, Ridge, and Lasso regression.
- Interpret how regularization changes coefficient size and model complexity.


## 1. Setup
### Packages used in this lab

In this lab, we will use a small set of Python packages for numerical computation, data handling, visualization, model fitting, and evaluation. These packages were introduced in earlier labs, so we will use them here without going into technical detail.

- `warnings` to suppress selected warning messages so the notebook output stays easier to read
- `NumPy` for numerical computation
- `pandas` for working with tabular data
- `Matplotlib` for visualization
- `scikit-learn` for loading the dataset, splitting the data, creating transformed features, standardizing predictors, fitting regression models, and evaluating model performance


In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning

SEED = 2026
np.random.seed(SEED)

warnings.filterwarnings("ignore", category=ConvergenceWarning)

## 2. Load the data

In this lab, we will continue using the diabetes dataset that was introduced in Module 3.


In [ ]:
diabetes = load_diabetes()

X = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y = pd.Series(diabetes.target, name="disease_progression")

print("Feature matrix shape:", X.shape)
print("Outcome shape:", y.shape)
X.head()

## 3. Train, validation, and test split

We divide the data into three parts:

- The training set is used to fit the model
- The validation set is used for model comparison and tuning
- The test set is used only for final evaluation



In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=SEED
)

print("Training set size:", X_train.shape[0])
print("Validation set size:", X_val.shape[0])
print("Test set size:", X_test.shape[0])

## 4. Overfitting with a more flexible model

To make overfitting easier to see, we will use three real predictors: `bmi`, `bp`, and `s5`.

We will fit polynomial regression models with increasing degree. As the degree increases, the model becomes more flexible and can fit the training data more closely. We will compare training, validation, and test RMSE to see when added flexibility stops helping and begins to hurt generalization.


In [ ]:
selected_features = ["bmi", "bp", "s5"]

x_train_small = X_train[selected_features]
x_val_small = X_val[selected_features]
x_test_small = X_test[selected_features]

degrees = [1, 2, 3, 5]
poly_rows = []

for degree in degrees:
    model = Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("scaler", StandardScaler()),
        ("linreg", LinearRegression())
    ])

    model.fit(x_train_small, y_train)
    n_features = model.named_steps["poly"].transform(x_train_small).shape[1]

    poly_rows.append({
        "degree": degree,
        "n_features": n_features,
        "train_rmse": np.sqrt(mean_squared_error(y_train, model.predict(x_train_small))),
        "validation_rmse": np.sqrt(mean_squared_error(y_val, model.predict(x_val_small))),
        "test_rmse": np.sqrt(mean_squared_error(y_test, model.predict(x_test_small))),
    })

poly_results = pd.DataFrame(poly_rows)
poly_results


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(poly_results["degree"], poly_results["train_rmse"], marker="o", label="Train RMSE")
plt.plot(poly_results["degree"], poly_results["validation_rmse"], marker="o", label="Validation RMSE")
plt.plot(poly_results["degree"], poly_results["test_rmse"], marker="o", label="Test RMSE")
plt.xlabel("Polynomial degree")
plt.ylabel("RMSE")
plt.title("Model flexibility and generalization")
plt.legend()
plt.show()


### Interpretation

In this example, increasing the polynomial degree lowers the training RMSE, which means the model is fitting the training data more closely.

However, after a point, the validation and test RMSE begin to rise. This is the pattern we expect under overfitting: the model is becoming more flexible, but that added flexibility is not improving generalization.

This provides one practical view of the bias-variance tradeoff:

- very simple models may underfit and have high bias
- very flexible models may overfit and have high variance
- useful predictive models usually balance the two


## 5. Why regularization?


The previous example used a small subset of predictors to show overfitting clearly.

Now we return to the full dataset and intentionally create a richer feature space using all predictors plus degree-2 polynomial terms. This creates many more coefficients to estimate and makes overfitting more likely.

We will compare:

- ordinary least squares
- Ridge regression, which adds an $L_2$ penalty
- Lasso regression, which adds an $L_1$ penalty

For ordinary least squares, we minimize the residual sum of squares:
$
\min_{\beta_0,\beta_1,\ldots,\beta_p}
\sum_{i=1}^n
\left(
y_i - \beta_0 - \sum_{j=1}^p x_{ij}\beta_j
\right)^2
$

Ridge regression adds an $L_2$ penalty:
$
\min_{\beta_0,\beta_1,\ldots,\beta_p}
\left\{
\sum_{i=1}^n
\left(
y_i - \beta_0 - \sum_{j=1}^p x_{ij}\beta_j
\right)^2
+
\lambda \sum_{j=1}^p \beta_j^2
\right\}
$

Lasso regression adds an $L_1$ penalty:
$
\min_{\beta_0,\beta_1,\ldots,\beta_p}
\left\{
\sum_{i=1}^n
\left(
y_i - \beta_0 - \sum_{j=1}^p x_{ij}\beta_j
\right)^2
+
\lambda \sum_{j=1}^p |\beta_j|
\right\}
$

In both cases, $\lambda$ controls how strongly large coefficients are penalized. Larger values of $\lambda$ usually shrink coefficients more strongly, which can reduce overfitting and improve generalization.


In [ ]:
poly = PolynomialFeatures(degree=2, include_bias=False)

X_train_poly = poly.fit_transform(X_train)
X_val_poly = poly.transform(X_val)
X_test_poly = poly.transform(X_test)

print("Original number of predictors:", X_train.shape[1])
print("Number of predictors after degree-2 expansion:", X_train_poly.shape[1])

### Ordinary least squares on the expanded feature set

We now fit an ordinary least squares model after adding degree-$2$ polynomial and interaction terms to all predictors.




In [ ]:
ols_poly = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

ols_poly.fit(X_train_poly, y_train)

ols_poly_results = pd.DataFrame({
    "set": ["train", "validation", "test"],
    "rmse": [
        np.sqrt(mean_squared_error(y_train, ols_poly.predict(X_train_poly))),
        np.sqrt(mean_squared_error(y_val, ols_poly.predict(X_val_poly))),
        np.sqrt(mean_squared_error(y_test, ols_poly.predict(X_test_poly))),
    ]
})

ols_poly_results

If the training error is much smaller than the validation and test errors, that is evidence that the expanded-feature OLS model is overfitting the training data.

### Tune Ridge and Lasso using the validation set

We will search over a grid of regularization strengths.

In our mathematical notation, this tuning parameter is written as $\lambda$. In `scikit-learn`, the corresponding argument is called `alpha`. Although the notation is different, they play the same role: larger values imply stronger regularization.

We will fit Ridge and Lasso models across a range of `alpha` values and select the value that gives the lowest validation RMSE.


In [ ]:
alphas = np.logspace(-2, 2, 20)

ridge_rows = []
lasso_rows = []

for alpha in alphas:
    ridge_model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=alpha, random_state=SEED))
    ])
    ridge_model.fit(X_train_poly, y_train)
    ridge_rows.append({
        "alpha": alpha,
        "train_rmse": np.sqrt(mean_squared_error(y_train, ridge_model.predict(X_train_poly))),
        "validation_rmse": np.sqrt(mean_squared_error(y_val, ridge_model.predict(X_val_poly))),
        "test_rmse": np.sqrt(mean_squared_error(y_test, ridge_model.predict(X_test_poly))),
    })

    lasso_model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Lasso(alpha=alpha, random_state=SEED, max_iter=100000))
    ])
    lasso_model.fit(X_train_poly, y_train)
    lasso_rows.append({
        "alpha": alpha,
        "train_rmse": np.sqrt(mean_squared_error(y_train, lasso_model.predict(X_train_poly))),
        "validation_rmse": np.sqrt(mean_squared_error(y_val, lasso_model.predict(X_val_poly))),
        "test_rmse": np.sqrt(mean_squared_error(y_test, lasso_model.predict(X_test_poly))),
        "nonzero_coefficients": np.sum(np.abs(lasso_model.named_steps["model"].coef_) > 1e-8)
    })

ridge_results = pd.DataFrame(ridge_rows)
lasso_results = pd.DataFrame(lasso_rows)

best_ridge_alpha = ridge_results.loc[ridge_results["validation_rmse"].idxmin(), "alpha"]
best_lasso_alpha = lasso_results.loc[lasso_results["validation_rmse"].idxmin(), "alpha"]

print("Best Ridge alpha:", best_ridge_alpha)
print("Best Lasso alpha:", best_lasso_alpha)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(ridge_results["alpha"], ridge_results["validation_rmse"], marker="o", label="Ridge validation RMSE")
plt.plot(lasso_results["alpha"], lasso_results["validation_rmse"], marker="o", label="Lasso validation RMSE")
plt.xscale("log")
plt.xlabel("alpha")
plt.ylabel("Validation RMSE")
plt.title("Validation performance across regularization strengths")
plt.legend()
plt.show()

### Fit the best Ridge and best Lasso models

In [ ]:
best_ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=best_ridge_alpha, random_state=SEED))
])

best_lasso = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Lasso(alpha=best_lasso_alpha, random_state=SEED, max_iter=100000))
])

best_ridge.fit(X_train_poly, y_train)
best_lasso.fit(X_train_poly, y_train)

comparison_table = pd.DataFrame([
    {
        "model": "OLS on degree-2 features",
        "train_rmse": np.sqrt(mean_squared_error(y_train, ols_poly.predict(X_train_poly))),
        "validation_rmse": np.sqrt(mean_squared_error(y_val, ols_poly.predict(X_val_poly))),
        "test_rmse": np.sqrt(mean_squared_error(y_test, ols_poly.predict(X_test_poly))),
    },
    {
        "model": "Best Ridge on degree-2 features",
        "train_rmse": np.sqrt(mean_squared_error(y_train, best_ridge.predict(X_train_poly))),
        "validation_rmse": np.sqrt(mean_squared_error(y_val, best_ridge.predict(X_val_poly))),
        "test_rmse": np.sqrt(mean_squared_error(y_test, best_ridge.predict(X_test_poly))),
    },
    {
        "model": "Best Lasso on degree-2 features",
        "train_rmse": np.sqrt(mean_squared_error(y_train, best_lasso.predict(X_train_poly))),
        "validation_rmse": np.sqrt(mean_squared_error(y_val, best_lasso.predict(X_val_poly))),
        "test_rmse": np.sqrt(mean_squared_error(y_test, best_lasso.predict(X_test_poly))),
    }
])

comparison_table


## 6. Coefficient shrinkage

Ridge usually keeps all coefficients but shrinks them toward zero.

Lasso can both shrink coefficients and set some of them exactly to zero, which is why it is often discussed as a variable selection method.

In [ ]:
feature_names_poly = poly.get_feature_names_out(X.columns)

ridge_coefs = pd.Series(best_ridge.named_steps["model"].coef_, index=feature_names_poly)
lasso_coefs = pd.Series(best_lasso.named_steps["model"].coef_, index=feature_names_poly)

coef_summary = pd.DataFrame({
    "ridge_abs_coef": ridge_coefs.abs(),
    "lasso_abs_coef": lasso_coefs.abs()
}).sort_values("ridge_abs_coef", ascending=False)

coef_summary.head(15)

In [ ]:
print("Number of nonzero Ridge coefficients:", np.sum(np.abs(ridge_coefs) > 1e-8))
print("Number of nonzero Lasso coefficients:", np.sum(np.abs(lasso_coefs) > 1e-8))

In [ ]:
top_features = coef_summary.head(12).index

plot_df = pd.DataFrame({
    "Ridge": ridge_coefs.loc[top_features],
    "Lasso": lasso_coefs.loc[top_features]
})

plot_df.plot(kind="bar", figsize=(10, 4))
plt.ylabel("Coefficient value")
plt.title("Selected coefficients from the best regularized models")
plt.xticks(rotation=45, ha="right")
plt.show()

## Key takeaways

1. A lower training error does not necessarily mean better generalization.
2. Increasing flexibility can reduce bias, but it can also increase variance and produce overfitting.
3. Regularization changes the objective function by penalizing large coefficients.
4. Ridge often improves stability through shrinkage.
5. Lasso adds shrinkage and can also remove predictors by setting some coefficients to zero.

## 7. Additional practice

1. Change the polynomial degree in the expanded feature set from degree $2$ to degree $3$. What happens to train, validation, and test error?
2. Repeat the Ridge and Lasso tuning with a wider grid of $\alpha$ values.
3. Use only a subset of clinical predictors and see whether the preferred model changes.
